# 02 · Modelo de Datos y Carga a Base de Datos
**Proyecto:** IW Resource Management – Caso Familia Miranda  
**Autor:** Diego Ballesteros  
**Fecha:** 2026

---

## Objetivo
Diseñar e implementar el modelo relacional que almacenará la información del caso de negocio, cumpliendo mínimo la **Tercera Forma Normal (3FN)**, e insertar los datos procesados en el notebook 01.

---

## Estructura del notebook
1. Justificación del modelo y decisiones de arquitectura
2. Diagrama del modelo relacional
3. Setup y conexión a la base de datos
4. Creación de tablas (DDL)
5. Inserción de datos
6. Validación de integridad

---
## 1. Justificación del modelo y decisiones de arquitectura

### ¿Por qué un modelo relacional?

El caso de negocio presenta información estructurada con relaciones claras entre entidades (miembros, rubros, gastos, presupuesto). Un modelo relacional es la elección correcta porque:

- **Integridad referencial**: garantiza que ningún gasto quede huérfano sin miembro o rubro válido.
- **Eliminación de redundancia**: los nombres de rubros y miembros se almacenan una sola vez y se referencian por ID, evitando inconsistencias.
- **Escalabilidad horizontal de datos**: agregar nuevos meses, miembros o familias requiere solo nuevas filas, no cambios de estructura.
- **Consultas complejas en SQL**: los reportes solicitados (comparativo planeado vs. real, top rubros excedidos, medio de pago preferido) se resuelven de forma natural y eficiente con JOIN y GROUP BY.

### Normalización hasta Tercera Forma Normal (3FN)

| Forma Normal | Regla | Cómo se cumple en este modelo |
|---|---|---|
| **1FN** | Valores atómicos, sin grupos repetidos | Cada celda tiene un valor único; los gastos de cada miembro están en filas separadas |
| **2FN** | Sin dependencias parciales (requiere 1FN) | Todas las tablas tienen PK simple (surrogate key); no hay PKs compuestas con dependencias parciales |
| **3FN** | Sin dependencias transitivas | `nombre_miembro` y `tipo_miembro` dependen únicamente de `id_miembro`; `nombre_rubro` depende solo de `id_rubro`; ningún atributo no-clave depende de otro atributo no-clave |

### Tecnología elegida: SQLite vía SQLAlchemy

- **SQLite**: base de datos embebida, sin servidor, cero configuración. El archivo `.db` es portable y reproducible — cualquier evaluador puede correr el notebook sin infraestructura adicional.
- **SQLAlchemy**: ORM estándar de la industria en Python. Permite definir el esquema de forma declarativa y hace la inserción segura contra inyección SQL.
- **Decisión de escalabilidad**: si el sistema creciera a múltiples familias o se requiriera concurrencia, la migración a PostgreSQL o Azure SQL Database sería directa — SQLAlchemy abstrae el motor subyacente con cambio mínimo de código (solo la `connection_string`).

### Arquitectura por capas (orientación Senior)

Para un sistema productivo con múltiples familias se plantearían tres capas:

```
┌─────────────────────────────────────────────┐
│  CAPA RAW (Bronze)                          │
│  Archivos fuente sin modificar              │
│  → data/raw/                                │
├─────────────────────────────────────────────┤
│  CAPA PROCESADA (Silver)                    │
│  Datos limpios y estandarizados             │
│  → data/processed/ (Parquet)                │
├─────────────────────────────────────────────┤
│  CAPA ANALÍTICA (Gold)                      │
│  Modelo relacional listo para reportes      │
│  → SQLite / PostgreSQL / Azure SQL          │
└─────────────────────────────────────────────┘
```

Esta arquitectura medallón (Bronze → Silver → Gold) es el estándar en ingeniería de datos moderna y garantiza trazabilidad completa desde el dato crudo hasta el reporte final.

---
## 2. Diagrama del modelo relacional

```
┌──────────────────┐         ┌───────────────────────────────────────┐
│    miembros      │         │               gastos                  │
│──────────────────│         │───────────────────────────────────────│
│ PK id_miembro    │──┐      │ PK id_gasto         INT  AUTOINCREMENT│
│    nombre        │  └─────>│ FK id_miembro       INT  NOT NULL     │
│    tipo          │         │ FK id_rubro         INT               │
└──────────────────┘         │    fecha            DATE NOT NULL     │
                             │    descripcion      TEXT              │
┌──────────────────┐         │    valor            REAL NOT NULL     │
│     rubros       │         │    forma_pago       TEXT              │
│──────────────────│         │    categoria_origen TEXT              │
│ PK id_rubro      │──┐      │    mes              TEXT NOT NULL     │
│    nombre_rubro  │  └─────>│ FK id_rubro         INT               │
└──────────────────┘         └───────────────────────────────────────┘
         │
         │              ┌────────────────────────────────┐
         │              │          presupuesto           │
         │              │────────────────────────────────│
         │              │ PK id_presupuesto  INT         │
         └─────────────>│ FK id_rubro        INT NOT NULL│
                        │    mes             TEXT NOT NULL│
                        │    valor_planeado  REAL NOT NULL│
                        │    UNIQUE(id_rubro, mes)        │
                        └────────────────────────────────┘
```

**Decisiones de diseño clave:**
- `gastos.id_rubro` es nullable — permite registrar gastos sin rubro presupuestado sin violar integridad.
- `presupuesto` tiene constraint `UNIQUE(id_rubro, mes)` — garantiza un solo valor planeado por rubro por mes.
- `miembros.tipo` distingue entre `'ingreso'` y `'gasto'` para facilitar el análisis de flujo de caja.

---
## 3. Setup y conexión a la base de datos

In [1]:
import pandas as pd
from pathlib import Path
from sqlalchemy import (
    create_engine, text,
    Column, Integer, String, Float, Date, UniqueConstraint, ForeignKey
)
from sqlalchemy.orm import declarative_base, Session
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path('..')
PROC_DIR = BASE_DIR / 'data' / 'processed'
DB_PATH  = BASE_DIR / 'data' / 'familia_miranda.db'

engine = create_engine(f'sqlite:///{DB_PATH}', echo=False)
Base   = declarative_base()

print(f'Motor  : {engine.dialect.name}')
print(f'Archivo: {DB_PATH.resolve()}')


Motor  : sqlite
Archivo: /home/diego/iw-proyecto/data/familia_miranda.db


---
## 4. Creación de tablas (DDL)

Se definen las tablas usando el ORM declarativo de SQLAlchemy. Esto es equivalente a ejecutar el DDL en SQL pero con la ventaja de que el esquema queda versionado en Python y es independiente del motor de base de datos.

In [2]:
class Miembro(Base):
    """Miembros de la familia Miranda."""
    __tablename__ = 'miembros'

    id_miembro = Column(Integer, primary_key=True, autoincrement=True)
    nombre     = Column(String(50),  nullable=False, unique=True)
    tipo       = Column(String(20),  nullable=False)


class Rubro(Base):
    """Rubros del presupuesto familiar."""
    __tablename__ = 'rubros'

    id_rubro     = Column(Integer, primary_key=True, autoincrement=True)
    nombre_rubro = Column(String(100), nullable=False, unique=True)


class Presupuesto(Base):
    """
    Valor planeado por rubro y mes.
    UNIQUE(id_rubro, mes) garantiza un solo valor planeado por rubro por mes.
    """
    __tablename__  = 'presupuesto'
    __table_args__ = (UniqueConstraint('id_rubro', 'mes', name='uq_rubro_mes'),)

    id_presupuesto = Column(Integer, primary_key=True, autoincrement=True)
    id_rubro       = Column(Integer, ForeignKey('rubros.id_rubro'), nullable=False)
    mes            = Column(String(7),  nullable=False)
    valor_planeado = Column(Float,      nullable=False)


class Gasto(Base):
    """
    Transacciones de gasto de cada miembro.
    id_rubro es nullable para permitir gastos sin rubro presupuestado.
    """
    __tablename__ = 'gastos'

    id_gasto         = Column(Integer, primary_key=True, autoincrement=True)
    id_miembro       = Column(Integer, ForeignKey('miembros.id_miembro'), nullable=False)
    id_rubro         = Column(Integer, ForeignKey('rubros.id_rubro'),     nullable=True)
    fecha            = Column(Date,    nullable=False)
    descripcion      = Column(String(200))
    valor            = Column(Float,   nullable=False)
    forma_pago       = Column(String(30))
    categoria_origen = Column(String(100))
    mes              = Column(String(7),  nullable=False)


Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

print('Tablas creadas:')
with engine.connect() as conn:
    tablas = conn.execute(text("SELECT name FROM sqlite_master WHERE type='table'")).fetchall()
    for t in tablas:
        count = conn.execute(text(f'SELECT COUNT(*) FROM {t[0]}')).scalar()
        print(f'   {t[0]:<20} {count} filas')


Tablas creadas:
   miembros             0 filas
   rubros               0 filas
   presupuesto          0 filas
   gastos               0 filas


---
## 5. Inserción de datos

Se cargan los datos procesados del notebook 01 y se insertan respetando el orden de dependencias: primero las tablas de referencia (`miembros`, `rubros`) y luego las tablas transaccionales (`presupuesto`, `gastos`).

In [3]:
df_gastos = pd.read_csv(PROC_DIR / 'gastos_clean.csv', parse_dates=['fecha'])
df_ppto   = pd.read_csv(PROC_DIR / 'presupuesto_clean.csv')

print(f'Gastos cargados    : {len(df_gastos)} registros')
print(f'Presupuesto cargado: {len(df_ppto)} registros')


Gastos cargados    : 372 registros
Presupuesto cargado: 74 registros


In [4]:
class DatabaseLoader:
    """Carga los datos procesados en las tablas del modelo relacional."""

    def __init__(self, engine, df_gastos: pd.DataFrame, df_ppto: pd.DataFrame):
        self.engine      = engine
        self.df_gastos   = df_gastos
        self.df_ppto     = df_ppto
        self.miembro_ids: dict = {}
        self.rubro_ids:   dict = {}

    def load_all(self) -> None:
        """Ejecuta la carga completa respetando el orden de dependencias."""
        self._load_miembros()
        self._load_rubros()
        self._load_presupuesto()
        self._load_gastos()

    def _load_miembros(self) -> None:
        data = [
            {'nombre': 'papa', 'tipo': 'papa'},
            {'nombre': 'mama', 'tipo': 'mama'},
            {'nombre': 'hijo', 'tipo': 'hijo'},
        ]
        with Session(self.engine) as session:
            session.bulk_insert_mappings(Miembro, data)
            session.commit()
            self.miembro_ids = {m.nombre: m.id_miembro for m in session.query(Miembro).all()}
        print(f'Miembros insertados : {self.miembro_ids}')

    def _load_rubros(self) -> None:
        rubros_ppto     = set(self.df_ppto['rubro'].str.strip().unique())
        rubros_sin_ppto = set(
            self.df_gastos[self.df_gastos['rubro_presupuesto'].isna()]['id_categoria'].unique()
        )
        rubros_todos = sorted(rubros_ppto | rubros_sin_ppto)
        with Session(self.engine) as session:
            session.bulk_insert_mappings(Rubro, [{'nombre_rubro': r} for r in rubros_todos])
            session.commit()
            self.rubro_ids = {r.nombre_rubro: r.id_rubro for r in session.query(Rubro).all()}
        print(f'Rubros insertados   : {len(self.rubro_ids)}')

    def _load_presupuesto(self) -> None:
        records = []
        for _, row in self.df_ppto.iterrows():
            rubro_norm = row['rubro'].strip()
            if rubro_norm in self.rubro_ids:
                records.append({
                    'id_rubro'      : self.rubro_ids[rubro_norm],
                    'mes'           : row['mes'],
                    'valor_planeado': float(row['valor_planeado']),
                })
        with Session(self.engine) as session:
            session.bulk_insert_mappings(Presupuesto, records)
            session.commit()
        print(f'Presupuesto insertado: {len(records)} registros')

    def _load_gastos(self) -> None:
        records = []
        skipped = 0
        for _, row in self.df_gastos.iterrows():
            id_miembro = self.miembro_ids.get(row['miembro'])
            if id_miembro is None:
                skipped += 1
                continue
            rubro_nombre = row.get('rubro_presupuesto')
            if pd.isna(rubro_nombre):
                rubro_nombre = row['id_categoria']
            id_rubro = self.rubro_ids.get(rubro_nombre)
            records.append({
                'id_miembro'      : id_miembro,
                'id_rubro'        : id_rubro,
                'fecha'           : row['fecha'].date() if pd.notna(row['fecha']) else None,
                'descripcion'     : str(row['descripcion'])[:200],
                'valor'           : float(row['valor']),
                'forma_pago'      : str(row['forma_pago']),
                'categoria_origen': str(row['id_categoria']),
                'mes'             : row['mes_origen'],
            })
        with Session(self.engine) as session:
            session.bulk_insert_mappings(Gasto, records)
            session.commit()
        print(f'Gastos insertados   : {len(records)}')
        if skipped:
            print(f'Omitidos            : {skipped} (miembro no reconocido)')

    def validate(self) -> None:
        """Verifica conteos e integridad referencial despues de la carga."""
        print('CONTEOS POR TABLA')
        print('-' * 40)
        with self.engine.connect() as conn:
            for tabla in ['miembros', 'rubros', 'presupuesto', 'gastos']:
                n = conn.execute(text(f'SELECT COUNT(*) FROM {tabla}')).scalar()
                print(f'  {tabla:<20} {n:>5} filas')

        print()
        query = text("""
            SELECT m.nombre AS miembro, g.mes,
                   COUNT(*)     AS num_transacciones,
                   SUM(g.valor) AS total_gasto
            FROM gastos g
            JOIN miembros m ON g.id_miembro = m.id_miembro
            GROUP BY m.nombre, g.mes
            ORDER BY g.mes, m.nombre
        """)
        with self.engine.connect() as conn:
            df_check = pd.read_sql(query, conn)
        df_check['total_gasto'] = df_check['total_gasto'].map('{:,.0f}'.format)
        print('GASTOS POR MIEMBRO Y MES')
        print('-' * 55)
        print(df_check.to_string(index=False))

        print()
        print('INTEGRIDAD REFERENCIAL')
        print('-' * 50)
        with self.engine.connect() as conn:
            n_huerfanos = conn.execute(text("""
                SELECT COUNT(*) FROM gastos g
                LEFT JOIN miembros m ON g.id_miembro = m.id_miembro
                WHERE m.id_miembro IS NULL
            """)).scalar()
            n_sin_rubro = conn.execute(text(
                "SELECT COUNT(*) FROM gastos WHERE id_rubro IS NULL"
            )).scalar()
            n_rubros_sin_gasto = conn.execute(text("""
                SELECT COUNT(DISTINCT p.id_rubro)
                FROM presupuesto p
                LEFT JOIN gastos g ON p.id_rubro = g.id_rubro AND p.mes = g.mes
                WHERE g.id_gasto IS NULL
            """)).scalar()
        estado = 'OK' if n_huerfanos == 0 else 'ERROR'
        print(f'  [{estado}] Gastos sin miembro valido     : {n_huerfanos} (esperado: 0)')
        print(f'  [INFO] Gastos sin rubro presupuestado: {n_sin_rubro}')
        print(f'  [INFO] Rubros sin ningun gasto        : {n_rubros_sin_gasto}')


In [5]:
db_loader = DatabaseLoader(engine, df_gastos, df_ppto)
db_loader.load_all()


Miembros insertados : {'papa': 1, 'mama': 2, 'hijo': 3}
Rubros insertados   : 44
Presupuesto insertado: 74 registros
Gastos insertados   : 372


---
## 6. Validación de integridad

Después de cada carga masiva es obligatorio verificar que los datos quedaron consistentes. Se ejecutan validaciones contra las reglas del modelo.

In [6]:
db_loader.validate()


CONTEOS POR TABLA
----------------------------------------
  miembros                 3 filas
  rubros                  44 filas
  presupuesto             74 filas
  gastos                 372 filas

GASTOS POR MIEMBRO Y MES
-------------------------------------------------------
miembro     mes  num_transacciones total_gasto
   mama 2023-08                 27  50,425,697
   papa 2023-08                117  61,887,898
   hijo 2023-09                 56   5,677,360
   mama 2023-09                 46  12,207,858
   papa 2023-09                126  25,187,628

INTEGRIDAD REFERENCIAL
--------------------------------------------------
  [OK] Gastos sin miembro valido     : 0 (esperado: 0)
  [INFO] Gastos sin rubro presupuestado: 0
  [INFO] Rubros sin ningun gasto        : 12
